# ACLing: fixed-label-budget experiment

Run this notebook on the real image datasets. It reruns the completed budget study, with no additional historical validation labels used for prompt selection or adapter tuning. Defaults are B=4,8,16,32 labels per class, 30 draws, and three checkpoints. Macro-F1 is the primary metric. Archived measurements are included in `evidence.zip`.

Historical splits and prompts make this a retrospective benchmark extension. Independent authors and external images remain separate evidence requirements. The supplied author forms are intentionally blank. Select a GPU runtime and run cells in order.


In [ ]:
from pathlib import Path
from google.colab import files
import subprocess, sys, zipfile, json, hashlib, io
print('Upload the repository ZIP')
uploaded = files.upload()
archive = next((Path(p) for p in uploaded if p.lower().endswith('.zip')), None)
assert archive is not None, 'A reproduction ZIP is required'
with zipfile.ZipFile(archive) as z:
    for member in z.infolist():
        assert (Path('/content')/member.filename).resolve().is_relative_to(Path('/content')), 'Unsafe ZIP member'
    z.extractall('/content')
REPO = Path('/content/waste-language-vision-reproduction')
assert (REPO/'scripts/run_budget_experiment.py').is_file(), 'Use the latest reproduction ZIP'
if (REPO/'evidence.zip').is_file() and not (REPO/'measurements/COMPRESSED_FILES.json').exists():
    with zipfile.ZipFile(REPO/'evidence.zip') as z:
        z.extractall(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO/'requirements.txt')], cwd=REPO, check=True)
subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=REPO, check=True)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Set paths and freeze the study

Edit the two dataset paths. Use a fresh STUDY_NAME for each completed, failed, or changed study. Leave optional prompt/external/group manifest fields blank for the first historical-grid budget run. Use the independent prompt CSVs only after the author forms are genuinely completed. Never change settings in response to test scores and present that as a prospectively frozen study.

The first run creates content-bound feature bundles. If images or prompt strings change, choose a new FEATURES_ROOT. Model inference is performed only when a matching bundle is absent. Historical core results can remain in their old folders.


In [ ]:
DATA = {
    'realwaste': Path('/content/drive/MyDrive/RealWaste'),  # EDIT
    'trashnet': Path('/content/drive/MyDrive/dataset-resized'),  # EDIT
}
STUDY_NAME = 'budget_historical_seed42_v1'
FEATURES_ROOT = Path('/content/acling_budget_features_v1')
BUDGETS = [4, 8, 16, 32]
REPEATS = 30
BACKBONES = ['clip', 'siglip', 'openclip']
PRIMARY_METRIC = 'macro_f1'
PROMPTS = {'realwaste': '', 'trashnet': ''}
GROUP_SPLITS = {'realwaste': '', 'trashnet': ''}
EXTERNAL = {'realwaste': '', 'trashnet': ''}
RUN_IMAGE_AUDIT = True
for path in DATA.values(): assert path.is_dir(), path
STUDY = REPO/'budget_results'/STUDY_NAME
STUDY.mkdir(parents=True, exist_ok=True)


In [ ]:
if str(REPO/'src') not in sys.path: sys.path.insert(0, str(REPO/'src'))
from wastevlm.core import classes_for
from wastevlm.budget_features import prompt_inventory
settings = dict(study=STUDY_NAME, budgets=BUDGETS, repeats=REPEATS, backbones=BACKBONES,
                primary_metric=PRIMARY_METRIC, seed=42, extra_validation_labels=0,
                primary_comparison='target_selected_adapter minus pooled_ensemble_adapter',
                exact_overlap_policy='Regroup source exact-pixel matches with seed 42 before evaluation; stop on external overlap',
                group_splits={}, external_manifests={}, prompt_sha256={})
for dataset in DATA:
    settings['prompt_sha256'][dataset] = prompt_inventory(classes_for(dataset), PROMPTS[dataset])[2]
    settings['group_splits'][dataset] = hashlib.sha256(Path(GROUP_SPLITS[dataset]).read_bytes()).hexdigest() if GROUP_SPLITS[dataset] else 'historical_seed42'
    settings['external_manifests'][dataset] = hashlib.sha256(Path(EXTERNAL[dataset]).read_bytes()).hexdigest() if EXTERNAL[dataset] else None
settings['code_sha256'] = {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in
    [REPO/'src/wastevlm/budget.py', REPO/'src/wastevlm/budget_features.py', REPO/'scripts/run_budget_experiment.py', REPO/'scripts/summarize_budget_experiment.py']}
plan = STUDY/'frozen_plan.json'
if plan.exists():
    assert json.loads(plan.read_text()) == settings, 'Settings changed. Use a new STUDY_NAME.'
else:
    plan.write_text(json.dumps(settings, indent=2))
print(json.dumps(settings, indent=2))


## Screen image overlap

The audit screens every image pair using exact file/pixel hashes and coarse perceptual similarity. Perceptual candidates still require actual image review; the screen is not exhaustive for shared scenes.

The frozen cleaning rule is to regroup exact source matches before evaluation when they cross partitions. This creates a new source split using seed 42, preserves the old manifest, records the resolved split hash, and changes no image labels. It is a retrospective group-aware partition, not an independent external dataset. Confirmed source/external overlap stops the notebook and requires a corrected external collection.


In [ ]:
import pandas as pd
if RUN_IMAGE_AUDIT:
    for dataset, data in DATA.items():
        manifest = GROUP_SPLITS[dataset] or str(REPO/f'reference/splits/{dataset}.csv')
        audit_out = STUDY/'image_audit'/dataset
        command = [sys.executable, 'scripts/audit_image_overlap.py', '--data', str(data), '--manifest', manifest, '--out', str(audit_out)]
        if EXTERNAL[dataset]: command += ['--external-manifest', EXTERNAL[dataset]]
        subprocess.run(command, cwd=REPO, check=True)
        candidates = pd.read_csv(audit_out/'candidate_pairs.csv')
        exact = candidates[candidates.kind.isin(['exact_file', 'exact_pixels']) & candidates.cross_partition.astype(str).str.lower().eq('true')]
        display(candidates.head(20))
        if not exact.empty:
            external_overlap = exact.split_a.eq('external') | exact.split_b.eq('external')
            assert not external_overlap.any(), 'Exact source/external overlap found. Preserve the audit and correct the external collection before a new study.'
            derived_split = STUDY/'resolved_splits'/f'{dataset}.csv'
            subprocess.run([sys.executable, 'scripts/make_group_split.py', '--hashes', str(audit_out/'hash_manifest.csv'), '--out', str(derived_split), '--seed', '42'], cwd=REPO, check=True)
            GROUP_SPLITS[dataset] = str(derived_split)
            print('Using new exact-duplicate-group split:', derived_split)
resolved = {dataset: dict(group_split=GROUP_SPLITS[dataset] or None, split_sha256=hashlib.sha256(Path(GROUP_SPLITS[dataset]).read_bytes()).hexdigest() if GROUP_SPLITS[dataset] else hashlib.sha256((REPO/f'reference/splits/{dataset}.csv').read_bytes()).hexdigest()) for dataset in DATA}
(STUDY/'resolved_data_config.json').write_text(json.dumps(resolved, indent=2))


## Run the equal-budget experiment

Each method uses the same B labeled images per class. Four inner folds select settings within that pool; the final cache uses the full pool. The old validation partition is unused. The pooled-text baseline uses more prompt strings, so text computation is not matched even though labeled-image budgets are matched.

The runner protects nonempty output directories. If a run fails after writing partial outputs, preserve them, change STUDY_NAME, and restart using the matching feature folder. A short smoke run is a software check and should not be reported as the planned 30-draw study.


In [ ]:
for dataset, data in DATA.items():
    command = [sys.executable, 'scripts/run_budget_experiment.py', '--dataset', dataset, '--data', str(data),
               '--out', str(STUDY/dataset), '--features', str(FEATURES_ROOT), '--seed', '42',
               '--budgets', *map(str, BUDGETS), '--repeats', str(REPEATS), '--backbones', *BACKBONES, '--objective', PRIMARY_METRIC]
    if GROUP_SPLITS[dataset]: command += ['--split-manifest', GROUP_SPLITS[dataset]]
    else: command += ['--reference-split', str(REPO/f'reference/splits/{dataset}.csv')]
    if PROMPTS[dataset]: command += ['--prompt-csv', PROMPTS[dataset]]
    if EXTERNAL[dataset]: command += ['--external-manifest', EXTERNAL[dataset]]
    subprocess.run(command, cwd=REPO, check=True)


## Audit the saved decisions and summarize paired effects

The summary adjusts primary p values across the supplied primary cells using Holm's method. Confidence intervals are pointwise and conditional on fixed test images; they do not measure variation across new datasets, authors or splits. Keep negative and inconclusive comparisons.


In [ ]:
for dataset in DATA:
    subprocess.run([sys.executable, 'scripts/audit_budget_results.py', '--results', str(STUDY/dataset)], cwd=REPO, check=True)
subprocess.run([sys.executable, 'scripts/summarize_budget_experiment.py', '--results', str(STUDY), '--out', str(STUDY/'summary')], cwd=REPO, check=True)
contrasts = pd.read_csv(STUDY/'summary/paired_comparisons.csv')
display(contrasts[contrasts.primary])


## Download the evidence

The export includes predictions, inner-CV scores and decisions, sampled labels/folds, prompt strings, audits, and portable feature bundles. Feature paths are converted to relative image IDs. This allows the CV arithmetic to be rechecked without downloading model weights. Features do not independently prove that image labels or provenance are correct.

Retain the complete `ACLing_budget_results.zip` archive, including negative and inconclusive results.


In [ ]:
import numpy as np
output = Path('/content/ACLing_budget_results.zip')
with zipfile.ZipFile(output, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in STUDY.rglob('*'):
        if p.is_file(): z.write(p, Path('budget_study')/p.relative_to(STUDY))
    for dataset in DATA:
        for model in BACKBONES:
            source = FEATURES_ROOT/dataset/'source'/f'{model}.npz'
            with np.load(source, allow_pickle=False) as arrays:
                portable = {key: arrays[key].copy() for key in arrays.files}
            relative_paths = pd.read_csv(STUDY/dataset/'split_manifest.csv').path.to_numpy(dtype=str)
            assert len(relative_paths) == len(portable['features'])
            portable['paths'] = relative_paths
            buffer = io.BytesIO()
            np.savez_compressed(buffer, **portable)
            z.writestr(str(Path('feature_audit')/dataset/f'{model}.npz'), buffer.getvalue())
files.download(str(output))
